# 02 Collect Sources

Collect and audit the dynamic source artifacts used by the Marshfield Event Catalog: CORA coastal water level, ERA5 ocean waves, direct AORC SST rainfall members, and NWM soil-moisture context.


> **Stage Contract**
>
> Requires: region setup outputs, data_sources.yaml, Earthdata credentials where required
>
> Produces: CORA, ERA5 waves, AORC SST rainfall, NWM soil-moisture source artifacts
>
> Next: 03_build_event_catalog.ipynb

In [ ]:
import sys
from pathlib import Path
import importlib

import pandas as pd
from IPython.display import display

location_root = Path.cwd().parent
repo_root = location_root.parents[1]
src_root = repo_root / "src"
src_path = str(src_root)
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)
importlib.invalidate_caches()

import design_events.collect_sources.notebook as collect
collect = importlib.reload(collect)

source_skip_existing = True

config, paths = collect.load_runtime(location_root / "config.yaml")
display(collect.source_collection_runtime_summary(config, paths))


## Source Collection Plan

Review the configured source contracts before starting provider calls. This is the first place a stakeholder should see whether a source is required, optional, disabled, or ready for the full Marshfield production window.


In [ ]:
plan = collect.build_source_collection_plan(config, paths)
plan_table = collect.source_collection_plan_with_reuse_table(
    plan,
    paths,
    source_skip_existing=source_skip_existing,
)
display(plan_table)


## CORA Coastal Water Level

CORA provides the historical boundary water-level series for Marshfield surge-event fitting. The downstream Event Catalog expects the hourly MSL CSV at `paths["waterlevel_csv"]`.


In [ ]:
cora_path = paths["waterlevel_csv"]
if cora_path.exists():
    waterlevel = pd.read_csv(cora_path, parse_dates=["time"])
    display(pd.Series({"rows": len(waterlevel), "start": waterlevel["time"].min(), "end": waterlevel["time"].max(), "path": str(cora_path)}))
    display(waterlevel.head())
else:
    display(pd.Series({"status": "missing", "expected_path": str(cora_path)}))


## ERA5 Ocean Waves

Marshfield is wave-enabled. The configured Earth Data Hub ERA5 ocean-wave Zarr variables are staged into a local NetCDF for SnapWave/SFINCS coupling.


In [ ]:
waves = config["collection"].get("era5_waves", {})
display(pd.Series({
    "provider": waves.get("provider"),
    "bbox_wgs84": waves.get("bbox_wgs84"),
    "output": str(paths["era5_waves_nc"]),
    "exists": paths["era5_waves_nc"].exists(),
}))


## Stochastic Storm Transposition Region

The SST region is the configured AORC SST transposition polygon, not a notebook-specific hand draw. For Marshfield, `config.yaml` points to the 100 km candidate region around the study grid footprint, chosen during the AORC homogeneity review so the rainfall catalogue samples plausible coastal New England storms without drifting into a much broader regional climate.


In [ ]:
# Plot the configured AORC SST transposition region before pulling rainfall members.
def _location_path(value):
    path = Path(value)
    if path.is_absolute():
        return path
    if path.parts and path.parts[0] in {"data", "02_flood", "01_grid"}:
        return paths["location_root"] / path
    return paths["repo_root"] / path


def plot_sst_region(config, paths, zoom=9, basemap=True):
    import geopandas as gpd
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch

    try:
        import contextily as ctx
    except ImportError:
        ctx = None

    region_file = config["collection"]["aorc_sst"]["transposition_region"]["geometry_file"]
    footprint_file = config["grid_footprint"]["source"]
    region_path = _location_path(region_file)
    footprint_path = _location_path(footprint_file)

    region = gpd.read_file(region_path).to_crs("EPSG:4326")
    footprint = gpd.read_file(footprint_path).to_crs("EPSG:4326")
    region_web = region.to_crs(epsg=3857)
    footprint_web = footprint.to_crs(epsg=3857)

    fig, ax = plt.subplots(figsize=(8, 8))
    region_web.plot(ax=ax, facecolor="#d95f0226", edgecolor="#d95f02", linewidth=2.5)
    footprint_web.boundary.plot(ax=ax, color="#1b9e77", linewidth=2.0)

    xmin, ymin, xmax, ymax = region_web.total_bounds
    pad_x = (xmax - xmin) * 0.05
    pad_y = (ymax - ymin) * 0.05
    ax.set_xlim(xmin - pad_x, xmax + pad_x)
    ax.set_ylim(ymin - pad_y, ymax + pad_y)

    if basemap and ctx is not None:
        try:
            ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik, zoom=zoom, attribution_size=7)
        except Exception as exc:
            ax.text(0.01, 0.01, f"basemap unavailable: {type(exc).__name__}", transform=ax.transAxes, fontsize=8)

    ax.legend(
        handles=[
            Patch(facecolor="#d95f0226", edgecolor="#d95f02", label="Storm transposition region"),
            Line2D([0], [0], color="#1b9e77", linewidth=2.0, label="Study grid footprint"),
        ],
        loc="lower left",
    )
    ax.set_axis_off()
    ax.set_title("Marshfield stochastic storm transposition region")
    fig.tight_layout()
    return fig, ax


fig, ax = plot_sst_region(config, paths)


## Direct AORC SST Rainfall Members

The direct AORC SST collector scans the transposition region, ranks rolling storm windows, and writes the rainfall-member table used to pair precipitation with synthetic coastal water-level events.


In [ ]:
aorc_sst = config["collection"].get("aorc_sst", {})
display(pd.Series({
    "source": "direct_aorc_sst",
    "transposition_region": aorc_sst.get("transposition_region", {}).get("geometry_file"),
    "rainfall_members": str(paths["aorc_sst_rainfall_members_csv"]),
    "rainfall_members_exists": paths["aorc_sst_rainfall_members_csv"].exists(),
}))


## NWM Soil-Moisture Context

Marshfield has no meaningful streamflow boundary in the baseline, but selected NWM LDAS soil-moisture cells are retained for antecedent-condition pairing.


In [ ]:
display(collect.nwm_soil_moisture_source_summary(config, paths))


## Run Collection

Run each configured source inside this notebook and show progress as each provider finishes. This cell is configured for the full Marshfield production collection window, so it overwrites partial smoke artifacts instead of reusing them.


In [ ]:
# Collect configured sources and summarize the artifacts.
run_collection = True
collection_result_table = collect.run_collect(
    config,
    paths,
    plan,
    run_collection=run_collection,
    skip_existing=source_skip_existing,
)
display(collection_result_table)


## Readiness Audit

The readiness report records which artifacts are present and which source contracts still need attention before event-catalog construction.


In [ ]:
readiness_summary, gates = collect.source_collection_readiness_report(config, paths)
display(readiness_summary)
display(gates)


## Collected Data Overview

Plot the collected source data in the form that best matches each source: spatial footprints and sample points for geography, source-window bars for coverage, and time-series or distribution plots for the dynamic forcing records.


In [ ]:
# Set up small plotting helpers.
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


def _location_path(value):
    path = Path(value)
    if path.is_absolute():
        return path
    if path.parts and path.parts[0] in {"data", "02_flood", "01_grid"}:
        return paths["location_root"] / path
    return paths["repo_root"] / path


def _read_csv(path, **kwargs):
    path = Path(path)
    return pd.read_csv(path, **kwargs) if path.exists() else pd.DataFrame()


def _first_column(frame, names):
    return next((name for name in names if name in frame.columns), None)


### SST

Storm transposition targets are plotted against the configured SST region and study area.


In [ ]:
# Plot the SST region, study area, and rainfall transposition targets.
import geopandas as gpd

study_area = gpd.read_file(_location_path(config["grid_footprint"]["source"])).to_crs("EPSG:4326")
sst_region = gpd.read_file(
    _location_path(config["collection"]["aorc_sst"]["transposition_region"]["geometry_file"])
).to_crs("EPSG:4326")
rainfall = _read_csv(paths["aorc_sst_rainfall_members_csv"], parse_dates=["storm_start", "storm_end"])

lon_column = _first_column(rainfall, ["centroid_lon", "transposed_centroid_lon"])
lat_column = _first_column(rainfall, ["centroid_lat", "transposed_centroid_lat"])
value_column = _first_column(rainfall, ["max_precip_in", "mean_precip_in", "max", "mean"])

fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
sst_region.plot(ax=ax, facecolor="#f4a26133", edgecolor="#d95f02", linewidth=1.8)
study_area.boundary.plot(ax=ax, color="black", linewidth=1.2)

if not rainfall.empty and lon_column and lat_column:
    rainfall_points = gpd.GeoDataFrame(
        rainfall.dropna(subset=[lon_column, lat_column]),
        geometry=gpd.points_from_xy(rainfall[lon_column], rainfall[lat_column]),
        crs="EPSG:4326",
    )
    rainfall_points.plot(
        ax=ax,
        column=value_column,
        cmap="inferno_r",
        markersize=24,
        alpha=0.75,
        edgecolor="white",
        linewidth=0.15,
        legend=value_column is not None,
        legend_kwds={"label": "72h rainfall magnitude", "shrink": 0.62},
    )

ax.legend(
    handles=[
        Patch(facecolor="#f4a26133", edgecolor="#d95f02", label="AORC SST region"),
        Patch(facecolor="none", edgecolor="black", label="study area"),
        Line2D([0], [0], marker="o", color="none", markerfacecolor="#c51b7d", markersize=8, label="rainfall transposition targets"),
    ],
    loc="best",
)
ax.set_title("Collected source geography")
ax.set_xlabel("")
ax.set_ylabel("")
plt.show()


### CORA Boundary Water Level

Boundary water levels are plotted independently from the spatial source map.


In [ ]:
# Plot the collected CORA boundary water level.
waterlevel = _read_csv(paths["waterlevel_csv"], parse_dates=["time"])

if not waterlevel.empty:
    value_column = _first_column(waterlevel, ["value", "zeta", "water_level", "waterlevel"])
    fig, ax = plt.subplots(figsize=(10, 3), constrained_layout=True)
    waterlevel.set_index("time")[value_column].plot(ax=ax, color="#2166ac", linewidth=0.6)
    ax.set_title("CORA boundary water level")
    ax.set_xlabel("")
    ax.set_ylabel("m MSL")
    plt.show()


### NWM Soil Moisture

Soil moisture is summarized across configured NWM points and layers.


In [ ]:
# Plot monthly mean NWM soil moisture variables when available.
soil_moisture = _read_csv(paths["nwm_soil_moisture_csv"], parse_dates=["time"])
requested_soil_variables = list(config["collection"].get("nwm", {}).get("soil_moisture", {}).get("variables", []))
available_soil_variables = [name for name in requested_soil_variables if name in soil_moisture.columns]
legacy_soil_variable = _first_column(soil_moisture, ["SOIL_M", "soil_m", "soil_moisture"])
if legacy_soil_variable and legacy_soil_variable not in available_soil_variables:
    available_soil_variables.append(legacy_soil_variable)
missing_soil_variables = [name for name in requested_soil_variables if name not in soil_moisture.columns]

if not soil_moisture.empty and available_soil_variables:
    monthly = (
        soil_moisture.groupby("time")[available_soil_variables]
        .mean()
        .resample("MS")
        .mean()
    )
    fig, ax = plt.subplots(figsize=(10, 3.5), constrained_layout=True)
    monthly.plot(ax=ax, linewidth=0.9)
    ax.set_title("NWM soil moisture state")
    ax.set_xlabel("")
    ax.set_ylabel("soil moisture / saturation")
    ax.legend(title="variable", loc="best", fontsize=8)
    plt.show()

    display(pd.Series({
        "requested_variables": requested_soil_variables,
        "available_variables": available_soil_variables,
        "missing_variables": missing_soil_variables,
        "csv": str(paths["nwm_soil_moisture_csv"]),
    }, name="nwm_soil_moisture_status"))
elif paths["nwm_soil_moisture_csv"].exists():
    display(pd.Series({
        "requested_variables": requested_soil_variables,
        "available_variables": available_soil_variables,
        "missing_variables": missing_soil_variables,
        "csv": str(paths["nwm_soil_moisture_csv"]),
    }, name="nwm_soil_moisture_status"))


### AORC SST Rainfall

Rainfall members are shown as selected-event magnitude and distribution summaries.


In [ ]:
# Plot compact AORC SST rainfall member summaries.
rainfall = _read_csv(paths["aorc_sst_rainfall_members_csv"], parse_dates=["storm_start", "storm_end"])

if not rainfall.empty:
    max_column = _first_column(rainfall, ["max_precip_mm", "max_precip_in", "max"])
    mean_column = _first_column(rainfall, ["mean_precip_mm", "mean_precip_in", "mean"])
    if max_column is None or mean_column is None:
        raise KeyError("rainfall members need max and mean precipitation columns")
    unit = "mm"
    if "precip_units" in rainfall and rainfall["precip_units"].notna().any():
        unit = str(rainfall["precip_units"].dropna().iloc[0])
    elif str(max_column).endswith("_in"):
        unit = "in"
    fig, axes = plt.subplots(1, 2, figsize=(10, 3), constrained_layout=True)
    rainfall.plot.scatter(x="rank", y=max_column, ax=axes[0], color="#d95f02", s=16, alpha=0.7)
    axes[0].set_title("AORC SST rainfall member maxima")
    axes[0].set_xlabel("rank")
    axes[0].set_ylabel(f"precipitation, {unit}")
    rainfall[[mean_column, max_column]].plot.hist(ax=axes[1], bins=20, alpha=0.65)
    axes[1].set_title("Rainfall distribution")
    axes[1].set_xlabel(f"precipitation, {unit}")
    plt.show()


### ERA5 Wave Forcing

Wave variables are reduced to monthly spatial means for a compact quality check.


In [ ]:
# Plot compact ERA5 wave time series when available.
import xarray as xr

wave_path = Path(paths["era5_waves_nc"])

if wave_path.exists():
    with xr.open_dataset(wave_path) as ds:
        time_name = _first_column(pd.DataFrame(columns=list(ds.coords) + list(ds.dims)), ["valid_time", "time"])
        wave_vars = [name for name in ["swh", "pp1d", "mwd", "wdw"] if name in ds.data_vars]
        fig, axes = plt.subplots(len(wave_vars), 1, figsize=(10, 1.8 * len(wave_vars)), sharex=True, constrained_layout=True)
        axes = [axes] if len(wave_vars) == 1 else axes

        for ax, name in zip(axes, wave_vars):
            spatial_dims = [dim for dim in ds[name].dims if dim != time_name]
            series = ds[name].mean(dim=spatial_dims).resample({time_name: "MS"}).mean()
            series.to_pandas().plot(ax=ax, linewidth=0.8)
            ax.set_title(f"ERA5 {name}")
            ax.set_xlabel("")
        plt.show()
